<a href="https://colab.research.google.com/github/E-tech-coder/DataScienceCapstoneProject/blob/Michi_v2/approach5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature Engineering + Embeddings + Random Forest Model**

Although the project description suggests incorporating historical job information, the labeled CSV files do not contain temporal or person-level data. Therefore, assumptions such as increasing seniority over time cannot be learned from data without label leakage.

In [ ]:
!git clone https://github.com/E-tech-coder/DataScienceCapstoneProject.git

In [ ]:
%cd DataScienceCapstoneProject
!git checkout Michi_v2

In [ ]:
!ls

The files department.csv and seniority.csv are used for this machine learning model. The models for department and seniority are trained separately, resulting in two independent models. However, preprocessing, feature engineering and other preprocessing steps are applied uniformly for both models.

In [ ]:
import pandas as pd
from collections import Counter
import re
import numpy as np

#define files
df_sen = pd.read_csv("seniority.csv")
df_dep = pd.read_csv("department.csv")

In [ ]:
print("seniority columns:", df_sen.columns.tolist())
print("department columns:", df_dep.columns.tolist())

In [ ]:
print("\nSeniority class distribution:", Counter(df_sen["label"]))
print("Department class distribution:", Counter(df_dep["label"]))

# **Text Normalization**

In [ ]:
#remove german umlauts
GERMAN_MAP = str.maketrans({
    "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss",
    "Ä": "ae", "Ö": "oe", "Ü": "ue"
})

#pipeline
def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = text.translate(GERMAN_MAP)
    text = re.sub(r"[\/\-\|\&\(\)\[\],]", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [ ]:
#apply text normalization on data
df_sen["text_norm"] = df_sen["text"].apply(normalize_text)
df_dep["text_norm"] = df_dep["text"].apply(normalize_text)

# **Implementing Structure Features**

First, we implement simple numerical features that are language-independent. In particular, they help to reflect differences between short entry-level roles (‘analyst’) and longer, more complex role titles (‘senior manager corporate strategy’) and complement semantic embeddings in a meaningful way.
*   title_length_chars – number of characters
*   title_word_count – number of words
*   avg_word_length – average word length



In [ ]:
#defining features
def extract_structure_features(norm_text: str) -> dict:
    if pd.isna(norm_text):
        norm_text = ""

    words = norm_text.split()
    word_lengths = [len(w) for w in words] if words else [0]

    return {
        "title_length_chars": len(norm_text),
        "title_word_count": len(words),
        "avg_word_length": sum(word_lengths) / len(word_lengths)
    }

In [ ]:
#apply features to the data sets
def add_structure_features(df):
    feats = df["text_norm"].apply(extract_structure_features)
    feats_df = pd.DataFrame(list(feats))
    return pd.concat([df.reset_index(drop=True), feats_df], axis=1)

df_sen = add_structure_features(df_sen)
df_dep = add_structure_features(df_dep)

In [ ]:
#mid-term review
display(df_sen.head(3))
display(df_dep.head(3))

# **Implementing Keyword Features**

We also implement numerical keyword count features to achieve a higher probability of correct classification for obvious hits. The keyword list is taken from the extended rule-based model, as a high matched accuracy has already been achieved there.

In [ ]:
#defining keywords for seniority
SENIORITY_KEYWORDS = {
    "management": [
        "ceo","cfo","cto","cmo","chief","vice","president","owner",
        "gesellschafter","founder","cofounder","co-founder","prokurist",
        "prokuristin","unternehmensinhaber","gesch"],
    "director": [
        "director","executive director","global director","finance director","strategy director"],
    "lead": [
        "lead","leitung","leiter","head"],
    "senior": [
        "senior","sr"],
    "junior": [
        "intern","junior","trainee","student","referent","auszubild"]
}

In [ ]:
#defining keywords for department
DEPARTMENT_KEYWORDS = {
    "administrative": [
        "assistentin","assistenz","assistent","office","assistant",
        "sekret","verwaltung","bueroleiter","facility","administrador"],
    "business_development": [
        "business development","business developer",
        "new business","strategic partnerships","bd"],
    "consulting": [
        "consultant","berater","sap","dynamics","erp"],
    "customer_support": [
        "support","customer","supporter"],
    "information technology": [
        "software","developer","engineer","architect","devops","cloud",
        "data scientist","data engineer","network",
        "systems administrator","administrator","it"],
    "human resources": [
        "human","resources","ressource","hr","personal",
        "recruitment","talent","personalleiter"],
    "marketing": [
        "marketing","communication","communications",
        "kommunikation","messe","event"],
    "project_management": [
        "project","projektleiter","projektmanager",
        "projektmanagement","projektleitung","projects"],
    "purchasing": [
        "einkauf","purchas","eink"],
    "sales": [
        "sales","vertrieb","vertriebsleiter","salesforce"]
}

In [ ]:
#keyword counting with word boundaries
def count_keywords_regex(text_norm: str, keyword_dict: dict) -> dict:
    if pd.isna(text_norm):
        text_norm = ""

    feats = {}
    for group, keywords in keyword_dict.items():
        count = 0
        for kw in keywords:
            pattern = rf"\b{re.escape(kw)}\b"
            count += len(re.findall(pattern, text_norm))
        feats[f"{group}_keyword_count"] = count

    return feats

In [ ]:
#apply features on dataset
def add_keyword_features(df):
    sen_feats = df["text_norm"].apply(
        lambda t: count_keywords_regex(t, SENIORITY_KEYWORDS)
    )
    dep_feats = df["text_norm"].apply(
        lambda t: count_keywords_regex(t, DEPARTMENT_KEYWORDS)
    )

    sen_df = pd.DataFrame(list(sen_feats)).add_prefix("sen_")
    dep_df = pd.DataFrame(list(dep_feats)).add_prefix("dep_")

    return pd.concat([df.reset_index(drop=True), sen_df, dep_df], axis=1)

df_sen = add_keyword_features(df_sen)
df_dep = add_keyword_features(df_dep)

# **Implementing Multilingual Word Embeddings**

We use multilingual embeddings in the following, as they can capture semantic similarities between job titles regardless of language. Multilingual sentence embeddings were calculated for each normalised job title. These embeddings represent the semantic meaning of the title in a dense numerical vector space.

In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

The MiniLM-L12 v2 is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

In [ ]:
#Calculate embeddings for job titles
def compute_embeddings(texts, model):
    return model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

In [ ]:
#apply on both data sets
X_sen_emb = compute_embeddings(df_sen["text_norm"].tolist(), embedding_model)
X_dep_emb = compute_embeddings(df_dep["text_norm"].tolist(), embedding_model)

print(X_sen_emb.shape)
print(X_dep_emb.shape)

We have 384 dimensions. This is computationally intensive and may be partially redundant. We break the dimensions down to 50. However, we only fit PCA after the train-test split.

In [ ]:
from sklearn.decomposition import PCA

#pca dimension reduction
N_PCA_COMPONENTS = 50

pca = PCA(
    n_components=N_PCA_COMPONENTS,
    random_state=42
)

# **Seniority Model**

First, we will finalise the model for seniority.

**Train/Test Split**

Now we will split the data sets into train and test data. We will then only apply PCA to train, to avoid feature larning on test data and data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

#train/test split
y_sen = df_sen["label"].values

idx_train_sen, idx_test_sen = train_test_split(
    np.arange(len(df_sen)),
    test_size=0.2,
    random_state=42,
    stratify=y_sen
)

print("Train size:", len(idx_train_sen))
print("Test size:", len(idx_test_sen))

In [ ]:
#pca on train data & transforming train/test
N_PCA_COMPONENTS = 50
pca_sen = PCA(n_components=N_PCA_COMPONENTS, random_state=42)

X_sen_train_emb = X_sen_emb[idx_train_sen]
X_sen_test_emb  = X_sen_emb[idx_test_sen]

X_sen_train_pca = pca_sen.fit_transform(X_sen_train_emb)
X_sen_test_pca  = pca_sen.transform(X_sen_test_emb)

print("PCA train shape:", X_sen_train_pca.shape)
print("PCA test shape:", X_sen_test_pca.shape)
print("Explained variance (sum):", round(pca_sen.explained_variance_ratio_.sum(), 4))

With only 50 PCA components, around 85% of the total variance of the original 384-dimensional sentence embeddings can be retained, indicating efficient dimension reduction with high information content.

**Build feature matrix**

In [ ]:
#structure features
STRUCT_COLS = ["title_length_chars", "title_word_count", "avg_word_length"]

#keyword count columns
SEN_KEYWORD_COLS = [c for c in df_sen.columns if c.startswith("sen_") and c.endswith("_keyword_count")]

print("Num structure features:", len(STRUCT_COLS))
print("Num keyword features:", len(SEN_KEYWORD_COLS))

In [ ]:
#add features
X_sen_struct_kw = df_sen[STRUCT_COLS + SEN_KEYWORD_COLS].values

X_sen_train_struct_kw = X_sen_struct_kw[idx_train_sen]
X_sen_test_struct_kw  = X_sen_struct_kw[idx_test_sen]

print("Train structure + sen_kw:", X_sen_train_struct_kw.shape)
print("Test structure + sen_kw:", X_sen_test_struct_kw.shape)

In [ ]:
#add pca
X_sen_train = np.hstack([X_sen_train_struct_kw, X_sen_train_pca])
X_sen_test  = np.hstack([X_sen_test_struct_kw,  X_sen_test_pca])

print("Final X_train:", X_sen_train.shape)
print("Final X_test:", X_sen_test.shape)

In [ ]:
#prepare y_train and y_test
y_sen_train = y_sen[idx_train_sen]
y_sen_test  = y_sen[idx_test_sen]

print("y_train:", y_sen_train.shape, "y_test:", y_sen_test.shape)

**Train Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_sen = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [ ]:
#train model
rf_sen.fit(X_sen_train, y_sen_train)

**Evaluation**

In [ ]:
#predictions on test data
y_sen_pred = rf_sen.predict(X_sen_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", round(accuracy_score(y_sen_test, y_sen_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_sen_test, y_sen_pred))

In [ ]:
confusion_matrix(y_sen_test, y_sen_pred)

The model has an accuracy of just under 92% on the test dataset and a macro-F1 score of 88%, which is impressive despite the class imbalance. Junior has the lowest recall, which can probably be explained by the low number of occurrences. Director is almost perfect. Lead and senior are both stable. Management is solid, but occasionally confuses lead and senior. This is probably due to the semantics of some job titles (e.g. senior manager).

**Explainability**

In [ ]:
#assemble feature names
feature_names = (
    STRUCT_COLS +
    SEN_KEYWORD_COLS +
    [f"pca_emb_{i}" for i in range(X_sen_train_pca.shape[1])]
)

len(feature_names), X_sen_train.shape[1]

In [ ]:
import matplotlib.pyplot as plt

#feature importances
fi_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf_sen.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

#top 15 Features
top_n = 15
top_df = fi_df.head(top_n)

#plot
plt.figure(figsize=(8, 5))
plt.barh(top_df["feature"][::-1], top_df["importance"][::-1])
plt.xlabel("Feature importance")
plt.title("Top 15 Feature Importances – Seniority (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
#importance of feature groups
def feature_group(f):
    if f in STRUCT_COLS:
        return "structure"
    if f.startswith("sen_"):
        return "keywords"
    if f.startswith("pca_emb_"):
        return "embeddings"
    return "other"

fi_df["group"] = fi_df["feature"].apply(feature_group)

fi_df.groupby("group")["importance"].sum().sort_values(ascending=False)

The feature importance analysis shows very consistent and technically plausible model behaviour. The PCA-reduced embeddings make the largest contribution to the prediction with around 67%, confirming that the semantic meaning of the job title is the most important signal for seniority. In addition, the keyword features contribute significantly to the model decision with around 30% and provide clear, easily interpretable seniority indicators. The structural features play only a supporting role, accounting for just under 3%. Overall, the model shows a meaningful combination of implicit semantic information (embeddings) and explicit domain knowledge (keywords), which explains both the high prediction quality and the good explainability of the approach.

**Cosine similarity**

We prepare a lookup base from training data: embeddings, labels, and title texts.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

#training data (embeddings + labels + titel) for lookup
X_train_emb_lookup = X_sen_emb[idx_train_sen]
y_train_lookup = y_sen[idx_train_sen]
text_train_lookup = df_sen["text"].values[idx_train_sen]
text_train_norm_lookup = df_sen["text_norm"].values[idx_train_sen]

print(X_train_emb_lookup.shape, y_train_lookup.shape)

In [ ]:
def explain_with_cosine_similarity(
    query_emb: np.ndarray,
    top_k: int = 5
) -> pd.DataFrame:

    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)

    sims = cosine_similarity(query_emb, X_train_emb_lookup).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    return pd.DataFrame({
        "rank": range(1, top_k + 1),
        "similarity": sims[top_idx],
        "train_label": y_train_lookup[top_idx],
        "train_title": text_train_lookup[top_idx],
        "train_title_norm": text_train_norm_lookup[top_idx],
    })

In [ ]:
j = 0
test_row_idx = idx_test_sen[j]

query_title = df_sen.loc[test_row_idx, "text"]
true_label = df_sen.loc[test_row_idx, "label"]

#embedding for this title (from X_sen_emb)
query_emb = X_sen_emb[test_row_idx]

print("Query title:", query_title)
print("True label:", true_label)

explain_df = explain_with_cosine_similarity(query_emb, top_k=5)
explain_df

The job title was classified as Management because its semantic embedding is highly similar to multiple training examples labeled as Management. The cosine similarity to the top reference titles is 1.0, indicating an almost identical semantic representation. All nearest neighbors correspond to Geschäftsführer roles, providing strong evidence for the predicted seniority level.

In [ ]:
for j in range(len(idx_test_sen)):
    idx = idx_test_sen[j]
    if df_sen.loc[idx, "label"] == "Senior":
        print("Index:", j)
        print("Title:", df_sen.loc[idx, "text"])
        break

In [ ]:
j = 2
test_row_idx = idx_test_sen[j]

query_title = df_sen.loc[test_row_idx, "text"]
true_label = df_sen.loc[test_row_idx, "label"]

query_emb = X_sen_emb[test_row_idx]

print("Query title:", query_title)
print("True label:", true_label)

explain_df = explain_with_cosine_similarity(query_emb, top_k=5)
explain_df

The job title “Technical Sales Engineer” is classified as Senior because its embedding is most similar to multiple Senior-level Sales Engineer roles. The nearest neighbors are consistently technical and sales-oriented Senior positions, with cosine similarities up to 0.89. Although one management-related title appears among the top neighbors, the overall similarity distribution clearly supports the Senior classification.

# **Department Model**

Next up, we finalise the department model.

**Train/Test Split**

Now we will split the department data set into train and test data. We will then only apply PCA to train, to avoid feature larning on test data and data leakage.

In [ ]:
#train/test split
y_dep = df_dep["label"].values

idx_train_dep, idx_test_dep = train_test_split(
    np.arange(len(df_dep)),
    test_size=0.2,
    random_state=42,
    stratify=y_dep
)

print("Train size:", len(idx_train_dep))
print("Test size:", len(idx_test_dep))

In [ ]:
#pca train data & transforming train/test
N_PCA_COMPONENTS = 50
pca_dep = PCA(n_components=N_PCA_COMPONENTS, random_state=42)

X_dep_train_emb = X_dep_emb[idx_train_dep]
X_dep_test_emb  = X_dep_emb[idx_test_dep]

X_dep_train_pca = pca_dep.fit_transform(X_dep_train_emb)
X_dep_test_pca  = pca_dep.transform(X_dep_test_emb)

print("PCA train shape:", X_dep_train_pca.shape)
print("PCA test shape:", X_dep_test_pca.shape)
print("Explained variance (sum):", round(pca_dep.explained_variance_ratio_.sum(), 4))

With 50 PCA components, around 85% of the total variance (as in the seniority model) of the 384 dimensional sentence embeddings can be retained.

**Build feature matrix**

In [ ]:
#structure features
STRUCT_COLS = ["title_length_chars", "title_word_count", "avg_word_length"]

#keyword count columns department
DEP_KEYWORD_COLS = [c for c in df_dep.columns if c.startswith("dep_") and c.endswith("_keyword_count")]

print("Num structure features:", len(STRUCT_COLS))
print("Num department keyword features:", len(DEP_KEYWORD_COLS))

In [ ]:
#add features
X_dep_struct_kw = df_dep[STRUCT_COLS + DEP_KEYWORD_COLS].values

X_dep_train_struct_kw = X_dep_struct_kw[idx_train_dep]
X_dep_test_struct_kw  = X_dep_struct_kw[idx_test_dep]

print("Train struct+dep_kw:", X_dep_train_struct_kw.shape)
print("Test struct+dep_kw:", X_dep_test_struct_kw.shape)

In [ ]:
#add pca
X_dep_train = np.hstack([X_dep_train_struct_kw, X_dep_train_pca])
X_dep_test  = np.hstack([X_dep_test_struct_kw,  X_dep_test_pca])

print("Final X_dep_train:", X_dep_train.shape)
print("Final X_dep_test:", X_dep_test.shape)

In [ ]:
#prepare y_train and y_test
y_dep_train = y_dep[idx_train_dep]
y_dep_test  = y_dep[idx_test_dep]

print("y_train:", y_dep_train.shape, "y_test:", y_dep_test.shape)

**Train Random Forest**

In [ ]:
rf_dep = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [ ]:
rf_dep.fit(X_dep_train, y_dep_train)

**Evaluation**

In [ ]:
#predictions on test data
y_dep_pred = rf_dep.predict(X_dep_test)

In [ ]:
print("Accuracy:", round(accuracy_score(y_dep_test, y_dep_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_dep_test, y_dep_pred))

In [ ]:
confusion_matrix(y_dep_test, y_dep_pred)

The model has an accuracy of 95% for in-sample performance, which is very good and mainly attributable to the large classes (marketing, sales and IT). These are recognised very reliably, which is probably due to clear semantic titles and the large amount of training data. The macro F1 score of 78% is slightly lower. This is due to the smaller classes. The class imbalance of the data set shows its limitations here. The confusion matrix confirms that misclassifications are plausible in terms of content.

**Explainability**

In [ ]:
#assemble feature names
feature_names_dep = (
    STRUCT_COLS +
    DEP_KEYWORD_COLS +
    [f"pca_emb_{i}" for i in range(X_dep_train_pca.shape[1])]
)

len(feature_names_dep), X_dep_train.shape[1]

In [ ]:
#feature importance
fi_dep_df = (
    pd.DataFrame({
        "feature": feature_names_dep,
        "importance": rf_dep.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

#top 15 features
top_n = 15
top_df = fi_dep_df.head(top_n)

#plot
plt.figure(figsize=(8, 5))
plt.barh(top_df["feature"][::-1], top_df["importance"][::-1])
plt.xlabel("Feature importance")
plt.title("Top 15 Feature Importances – Department (Random Forest)")
plt.tight_layout()
plt.show()

In [ ]:
#importance of feature groups
def feature_group_dep(f):
    if f in STRUCT_COLS:
        return "structure"
    if f.startswith("dep_"):
        return "keywords"
    if f.startswith("pca_emb_"):
        return "embeddings"
    return "other"

fi_dep_df["group"] = fi_dep_df["feature"].apply(feature_group_dep)

fi_dep_df.groupby("group")["importance"].sum().sort_values(ascending=False)

The feature importance analysis indicates that the PCA-reduced embedding features contribute the largest share to the model’s predictions, accounting for approximately 63 % of the total importance. The remaining contribution is largely driven by the keyword-based features, while the structure-related features play only a minor role. This suggests that simple structural characteristics such as word length or word count have little influence on the department prediction.

Overall, the distribution of feature importances is relatively balanced, as no single feature dominates the model. The most influential individual feature accounts for only about 6 % of the total importance, indicating that the model relies on a combination of multiple signals rather than a single decisive feature.

**Cosine Similarity**

Here, too, we look at the cosine similarities.

In [ ]:
#training data for department explainability
X_dep_train_emb_lookup = X_dep_emb[idx_train_dep]
y_dep_train_lookup = y_dep_train
text_dep_train_lookup = df_dep["text"].values[idx_train_dep]
text_dep_train_norm_lookup = df_dep["text_norm"].values[idx_train_dep]

print(X_dep_train_emb_lookup.shape, y_dep_train_lookup.shape)

In [ ]:
def explain_department_with_cosine_similarity(
    query_emb: np.ndarray,
    top_k: int = 5
) -> pd.DataFrame:
    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)

    sims = cosine_similarity(query_emb, X_dep_train_emb_lookup).flatten()
    top_idx = sims.argsort()[::-1][:top_k]

    return pd.DataFrame({
        "rank": range(1, top_k + 1),
        "similarity": sims[top_idx],
        "train_label": y_dep_train_lookup[top_idx],
        "train_title": text_dep_train_lookup[top_idx],
        "train_title_norm": text_dep_train_norm_lookup[top_idx],
    })

In [ ]:
j = 0  #index in test set
test_row_idx = idx_test_dep[j]

query_title = df_dep.loc[test_row_idx, "text"]
true_label = df_dep.loc[test_row_idx, "label"]
pred_label = rf_dep.predict(X_dep_test[j].reshape(1, -1))[0]

query_emb = X_dep_emb[test_row_idx]

print("Query title:", query_title)
print("True label:", true_label)
print("Predicted label:", pred_label)

explain_department_with_cosine_similarity(query_emb, top_k=5)

The job title is classified as Sales because its semantic embedding is highly similar to multiple Sales-related leadership roles. The nearest neighbors consistently include titles such as Head of Sales, Sales Director and Sales Manager with cosine similarity values close to 0.9. All top reference titles are unambiguously associated with the Sales department, providing strong and consistent evidence for the predicted classification.

In [ ]:
#looking for consulting or project management
for j in range(len(idx_test_dep)):
    idx = idx_test_dep[j]
    if df_dep.loc[idx, "label"] in ["Consulting", "Project Management"]:
        print("j =", j, "| title:", df_dep.loc[idx, "text"])
        break

In [ ]:
j = 18

test_row_idx = idx_test_dep[j]
query_emb = X_dep_emb[test_row_idx]

explain_department_with_cosine_similarity(query_emb, top_k=5)

The job title is classified as Consulting because its semantic representation closely matches multiple Consulting roles in the training data. The nearest neighbors consistently include senior-level consultant positions, such as Senior Consultant and Senior Managing Consultant, with high cosine similarity values above 0.85. The uniformity of the neighboring labels provides strong evidence for the Consulting classification.

# **Conclusion**

Both models follow the same modular architecture, combining feature engineering, multilingual sentence embeddings, and Random Forest classifiers, while remaining fully interpretable and leakage-free.

From a performance perspective, both models achieve strong predictive results. The seniority model reaches an accuracy of approximately 92 %, with particularly reliable predictions for higher seniority levels such as Senior, Lead, Director, and Management. Most misclassifications occur between adjacent seniority levels (e.g. Senior vs. Lead), reflecting inherent ambiguity in job titles rather than systematic model errors.
The department model performs even stronger, achieving an accuracy of over 95 %, with excellent results for dominant classes such as Sales, Marketing, and Information Technology. Lower recall values for very small classes are primarily driven by data imbalance and semantic overlap with larger departments.

The explainability analysis confirms that the models behave in a conceptually sound and intuitive manner. Global feature-importance analysis shows that PCA-reduced embeddings contribute the largest share to both models’ decisions, followed by keyword-based features, while structural features play only a minor role.

Now, we will apply these two models on our out of sample test data set and have a look on their real world performance.

# **Evaluation Out-Of-Sample**

In [ ]:
#define test data set
df_oos = pd.read_csv("df_profiles_cleansed.csv")
print(df_oos.shape)
print(df_oos.columns.tolist())
df_oos.head(3)

We evaluate the trained seniority and department models on the out-of-sample data (df_oos). This involves preprocessing the out-of-sample data in the same way as the training data, making predictions, and then evaluating the performance against the actual labels in df_oos.


In [ ]:
#filter active positions only
df_oos_active = df_oos[df_oos['status'] == 'ACTIVE'].copy()
print(f"Number of active positions: {len(df_oos_active)}")

#normalize out-of-sample text
df_oos_active['text_norm'] = df_oos_active['position'].apply(normalize_text)

#extract structure features
def add_structure_features_oos(df):
    feats = df["text_norm"].apply(extract_structure_features)
    feats_df = pd.DataFrame(list(feats))
    return pd.concat([df.reset_index(drop=True), feats_df], axis=1)

df_oos_active = add_structure_features_oos(df_oos_active)

#extract keyword features
def add_keyword_features_oos(df):
    sen_feats = df["text_norm"].apply(lambda t: count_keywords_regex(t, SENIORITY_KEYWORDS))
    dep_feats = df["text_norm"].apply(lambda t: count_keywords_regex(t, DEPARTMENT_KEYWORDS))

    sen_df = pd.DataFrame(list(sen_feats)).add_prefix("sen_")
    dep_df = pd.DataFrame(list(dep_feats)).add_prefix("dep_")

    return pd.concat([df.reset_index(drop=True), sen_df, dep_df], axis=1)

df_oos_active = add_keyword_features_oos(df_oos_active)

#compute Embeddings
X_oos_emb = compute_embeddings(df_oos_active["text_norm"].tolist(), embedding_model)

#apply PCA to embeddings
X_oos_sen_pca = pca_sen.transform(X_oos_emb)
X_oos_dep_pca = pca_dep.transform(X_oos_emb)

#assemble feature matrices
#for Seniority
X_oos_sen_struct_kw = df_oos_active[STRUCT_COLS + SEN_KEYWORD_COLS].values
X_oos_sen = np.hstack([X_oos_sen_struct_kw, X_oos_sen_pca])
print(f"final X_oos_sen shape: {X_oos_sen.shape}")

#for Department
X_oos_dep_struct_kw = df_oos_active[STRUCT_COLS + DEP_KEYWORD_COLS].values
X_oos_dep = np.hstack([X_oos_dep_struct_kw, X_oos_dep_pca])
print(f"final X_oos_dep shape: {X_oos_dep.shape}")

#labels for evaluation on active positions
y_oos_sen_actual = df_oos_active['seniority'].values
y_oos_dep_actual = df_oos_active['department'].values

#predictions
y_oos_sen_pred = rf_sen.predict(X_oos_sen)
y_oos_dep_pred = rf_dep.predict(X_oos_dep)

#evaluation of seniority model
print("\n--- Out-of-Sample Seniority Model Evaluation ---")
print("Accuracy:", round(accuracy_score(y_oos_sen_actual, y_oos_sen_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_oos_sen_actual, y_oos_sen_pred, zero_division=0))

#evaluation of department model
print("\n--- Out-of-Sample Department Model Evaluation ---")
print("Accuracy:", round(accuracy_score(y_oos_dep_actual, y_oos_dep_pred), 4))
print("\nClassification Report:\n")
print(classification_report(y_oos_dep_actual, y_oos_dep_pred, zero_division=0))

#summary
print("\n--- Summary of Out-of-Sample Performance ---")
print(f"Seniority Model OOS Accuracy: {round(accuracy_score(y_oos_sen_actual, y_oos_sen_pred), 4)}")
print(f"Department Model OOS Accuracy: {round(accuracy_score(y_oos_dep_actual, y_oos_dep_pred), 4)}")


The out-of-sample performance of both models is substantially lower than their respective in-sample test accuracies. This points to a pronounced mismatch between the training data and the real-world dataset, particularly in terms of class definitions, label coverage, and class distributions. As a result, the available training data is not fully representative of real-world profiles, which limits the models ability to generalize.

Concretely, the seniority training set does not include the category Professional, which accounts for 38% of the out-of-sample dataset. Similarly, the department target includes an Other category that represents more than 56% of the out-of-sample data. Moreover, the class imbalance differs markedly between training and out-of-sample data (as also reflected in the alternative models developed in this project). Therefore, the real-world performance is primarily constrained by the training data rather than by the modeling approach itself.

For improved comparability, the labels Professional and Other are excluded from the evaluation.

In [ ]:
#fiter out 'Professional' for seniority evaluation
sen_mask = y_oos_sen_actual != 'Professional'
y_oos_sen_actual_filtered = y_oos_sen_actual[sen_mask]
y_oos_sen_pred_filtered = y_oos_sen_pred[sen_mask]

#evaluation of seniority model (excluding 'Professional')
print("\n--- Out-of-Sample Seniority Model Evaluation (Excluding Professional) ---")
print("Accuracy:", round(accuracy_score(y_oos_sen_actual_filtered, y_oos_sen_pred_filtered), 4))
print("\nClassification Report:\n")
print(classification_report(y_oos_sen_actual_filtered, y_oos_sen_pred_filtered, zero_division=0))

#filter out 'Other' for department evaluation
dep_mask = y_oos_dep_actual != 'Other'
y_oos_dep_actual_filtered = y_oos_dep_actual[dep_mask]
y_oos_dep_pred_filtered = y_oos_dep_pred[dep_mask]

#evaluation of department model (excluding 'Other')
print("\n--- Out-of-Sample Department Model Evaluation (Excluding Other) ---")
print("Accuracy:", round(accuracy_score(y_oos_dep_actual_filtered, y_oos_dep_pred_filtered), 4))
print("\nClassification Report:\n")
print(classification_report(y_oos_dep_actual_filtered, y_oos_dep_pred_filtered, zero_division=0))

#summary of filtered out-of-sample performance
print("\n--- Summary of Out-of-Sample Performance (Filtered) ---")
print(f"Seniority Model OOS Accuracy (Excluding Professional): {round(accuracy_score(y_oos_sen_actual_filtered, y_oos_sen_pred_filtered), 4)}")
print(f"Department Model OOS Accuracy (Excluding Other): {round(accuracy_score(y_oos_dep_actual_filtered, y_oos_dep_pred_filtered), 4)}")


**Short Analysis (Confusion Matrix plot)**

In [ ]:
import seaborn as sns

#get unique labels for seniority model
sen_labels = sorted(np.unique(np.concatenate((y_oos_sen_actual_filtered, y_oos_sen_pred_filtered))))

#calculate and print confusion matrix for seniority model
cm_sen_filtered = confusion_matrix(y_oos_sen_actual_filtered, y_oos_sen_pred_filtered, labels=sen_labels)

#plot confusion matrix for seniority
plt.figure(figsize=(8, 6))
sns.heatmap(cm_sen_filtered, annot=True, fmt='d', cmap='Blues', xticklabels=sen_labels, yticklabels=sen_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Seniority Model Confusion Matrix (Filtered Out-of-Sample)')
plt.show()

In [ ]:
#get unique labels for department model
dep_labels = sorted(np.unique(np.concatenate((y_oos_dep_actual_filtered, y_oos_dep_pred_filtered))))

#calculate and print confusion matrix for department model
cm_dep_filtered = confusion_matrix(y_oos_dep_actual_filtered, y_oos_dep_pred_filtered, labels=dep_labels)

#plot Confusion matrix for department
plt.figure(figsize=(10, 8))
sns.heatmap(cm_dep_filtered, annot=True, fmt='d', cmap='Blues', xticklabels=dep_labels, yticklabels=dep_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Department Model Confusion Matrix (Filtered Out-of-Sample)')
plt.show()

**Conclusion**

Seniority:

* Junior has a very low recall rate (≈ 0.21); only 3 out of 14 actual junior positions were correctly identified. Most were classified as senior.
* Senior has low precision (0.35). A large proportion of the roles predicted as senior actually belonged to neighbouring seniority levels such as junior, lead or management.



---


Department:
* Purchasing was not correctly recognised in any of the 16 cases (recall = 0.00). Most roles were classified as sales or information technology.
* Information Technology shows low precision (0.43). Several roles from Administrative, Business Development, Consulting and Project Management were incorrectly classified as IT.





The results show that misclassifications occur primarily in semantically similar classes. Better coverage of such borderline cases in the training data set and more precise, more distinctive key terms for the classes concerned could improve the results.

Further Steps:
*  Expand and rebalance training data: Improve coverage of missing and underrepresented classes (e.g. Professional, Purchasing) to better reflect real-world distributions.

*  Address class imbalance: Apply targeted data collection, re-sampling, or class-weighting strategies to improve recall for minority classes.

*  Harmonize label definitions: Reduce ambiguity between adjacent or overlapping classes through clearer labeling guidelines.

*  Refine features for edge cases: Strengthen class-specific keyword features to complement embedding-based representations.

*  Iterative model improvement: Incorporate misclassified real-world examples via incremental or active learning.